# Persistence test — army vs the rebounding institutions (SDL 2026, final fitting session)

**Run 2026-07-16.** Contribution 1's persistence claim ("the secular gradient collapsed everywhere
in 2001 and rebounded everywhere under AKP except the army") rested on an independence-SE Wald
(final_checks cell 15: army vs parliament LEVEL at WV5, p = 0.068) — a conservative upper bound,
since both estimates come from the same respondents. This session recovers the cross-outcome
covariance properly: a respondent-level cluster bootstrap (Check A, primary), a stacked
binary logit with respondent-clustered SEs (Check B, companion), and a weighted variant (Check C).
New files only: this notebook, `results/persistence_test_table.csv`.

**Terminology.** LEVEL = nr total effect at a wave (baseline + interaction) — the quantity the
claim is about. INTER = nr × wave change from the WV2 baseline — reported, but not comparable
across institutions (parliament's 1990 baseline was itself −0.89).

**Sample.** The n = 8,777 common sample of final_checks cell 15 = non-missing on
conf_military + conf_parliament + conf_government + all layer4b predictors. Note: cell 15
conditions on that trio, *not* on all six confidence outcomes — an all-six definition would
exclude WV4 entirely (courts not fielded in 2001) and cannot give n = 8,777. conf_civil_service
has 66 missing values inside the common sample; its fits use the n = 8,711 subset.

## Verdicts

1. **PRIMARY.** Army vs parliament LEVEL at WV5: **+0.398**, bootstrap SE **0.196** vs independence 0.218 (**≈10% smaller**; bootstrap corr of the two level estimates +0.25), percentile 95% CI **[+0.015, +0.793]**, **p = 0.042** — the old independence-SE p was 0.068. The WV5 contrast is significant once the cross-outcome covariance is recovered.
2. **JOINT PERSISTENCE.** Army vs parliament across WV5–WV7: **χ²(3) = 4.37, p = 0.224 → NOT SUPPORTED** — parliament's own secular gradient collapses again by 2018 (LEVEL −0.076, n.s.), so "the army never came back" does not separate army from parliament over the full window. It **does** separate army from government (**χ²(3) = 31.12, p < 10⁻⁵**) and from civil service (χ²(3) = 16.21, p = 0.001).
3. **AGREEMENT.** Significance: **one disagreement** — army vs parliament at WV5 is p = 0.042 (ordinal bootstrap) but p = 0.151 (clustered binary logit; 0.145 weighted): the binary high-confidence recode at a ~90% ceiling discards the response variation the ordinal model uses. Sign: agreement on eight of nine contrast × wave cells; the exception is army vs parliament at WV7, where both routes estimate zero (ordinal −0.004, binary +0.038; p = .94/.82) and the sign flip is noise around zero. All three joint tests agree in significance class across routes; weighting changes nothing.


In [1]:
# Setup, REPLICATION GATE, common sample, full-sample ordinal fits
import time, warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel
from scipy import optimize, stats
from scipy.special import expit

warnings.filterwarnings("ignore")
pd.set_option("display.width", 230)
RESULTS = "results"
WAVES = ["WV2", "WV3", "WV4", "WV5", "WV6", "WV7"]
LATER = WAVES[1:]
W567 = ["WV5", "WV6", "WV7"]
INSTS = ["conf_military", "conf_parliament", "conf_government", "conf_civil_service"]
SHORT = {i: i.replace("conf_", "") for i in INSTS}

raw = pd.read_csv(f"{RESULTS}/pooled_extended_v2.csv")
df = raw.dropna(subset=["cluster"]).copy()
df["not_religious_dummy"] = df["rel_person"].map(
    lambda v: 1 if v in (2, 3) else (0 if v == 1 else np.nan))
for w in LATER:
    df[f"wave_{w}"] = (df["wave"] == w).astype(int)
    df[f"nr_x_{w}"] = df["not_religious_dummy"] * df[f"wave_{w}"]
PREDICTORS = ["not_religious_dummy", "rel_attendance", "rel_importance_god", "left_right",
              "age", "female", "education_raw", "income_raw", "married_dummy",
              ] + [f"wave_{w}" for w in LATER] + [f"nr_x_{w}" for w in LATER]

# GATE — exact layer4b army spec
subg = df[["conf_military"] + PREDICTORS].dropna().copy()
subg["conf_military"] = subg["conf_military"].astype(int)
rg = OrderedModel(subg["conf_military"], subg[PREDICTORS], distr="logit").fit(
    method="bfgs", maxiter=400, disp=False)
pub = pd.read_csv(f"{RESULTS}/layer4b_longwindow_interactions.csv")
prow = pub.query("institution=='conf_military' and predictor=='nr_x_WV4'").iloc[0]
g_c, g_s = float(rg.params["nr_x_WV4"]), float(rg.bse["nr_x_WV4"])
print(f"GATE refit nr_x_WV4 = {g_c:+.6f} (SE {g_s:.6f}) n={len(subg)} | "
      f"published {float(prow['coef']):+.6f} ({float(prow['se']):.6f}) n={int(prow['n_obs'])}")
assert abs(g_c - prow["coef"]) < 1e-4 and abs(g_s - prow["se"]) < 1e-4, "GATE FAILED — STOP"
assert len(subg) == int(prow["n_obs"]) == 8915, "GATE FAILED on n — STOP"
print("REPLICATION GATE PASSED")

# common sample (final_checks cell 15 definition)
TRIO = ["conf_military", "conf_parliament", "conf_government"]
common = df.dropna(subset=TRIO + PREDICTORS).reset_index(drop=True).copy()
assert len(common) == 8777
print(f"common sample n = {len(common)}; conf_civil_service non-missing = "
      f"{int(common['conf_civil_service'].notna().sum())}")
print("wave n:", dict(common["wave"].value_counts().reindex(WAVES)))

X_pred = common[PREDICTORS].to_numpy(dtype=float)
Y = {inst: common[inst].to_numpy(dtype=float) for inst in INSTS}
FULL = {}
for inst in INSTS:
    m = ~np.isnan(Y[inst])
    r = OrderedModel(pd.Series(Y[inst][m].astype(int)),
                     pd.DataFrame(X_pred[m], columns=PREDICTORS),
                     distr="logit").fit(method="bfgs", maxiter=400, disp=False)
    assert r.mle_retvals.get("converged", False), inst
    FULL[inst] = r

def level(res, w):
    L = np.zeros(len(res.params))
    L[res.params.index.get_loc("not_religious_dummy")] = 1
    L[res.params.index.get_loc(f"nr_x_{w}")] = 1
    return float(L @ res.params), float(np.sqrt(L @ res.cov_params() @ L))

ROWS = []
def add(method, contrast, wave, quantity, estimate, se=np.nan, ci=(np.nan, np.nan),
        p=np.nan, n_boot=np.nan, notes=""):
    ROWS.append({"method": method, "contrast": contrast, "wave": wave,
                 "quantity": quantity, "estimate": estimate, "se": se,
                 "ci_lo": ci[0], "ci_hi": ci[1], "p": p, "n_boot": n_boot, "notes": notes})

add("GATE", "army", "WV4", "INTER", g_c, g_s,
    p=float(rg.pvalues["nr_x_WV4"]), notes="layer4b refit = published (n=8915)")
print("\nLEVELs and INTERs (common sample, ordinal, delta-method SE):")
for inst in INSTS:
    r = FULL[inst]
    n_i = int((~np.isnan(Y[inst])).sum())
    parts = []
    for w in W567:
        c, se = level(r, w)
        pz = 2 * (1 - stats.norm.cdf(abs(c / se)))
        parts.append(f"{w} {c:+.4f}({se:.4f})")
        add("FULLSAMPLE_ORDINAL", SHORT[inst], w, "LEVEL", c, se, p=pz,
            notes=f"n={n_i}")
        ic, ise = float(r.params[f"nr_x_{w}"]), float(r.bse[f"nr_x_{w}"])
        add("FULLSAMPLE_ORDINAL", SHORT[inst], w, "INTER", ic, ise,
            p=float(r.pvalues[f"nr_x_{w}"]), notes=f"n={n_i}")
    print(f"  {SHORT[inst]:<14} n={n_i}  " + "  ".join(parts))

POINT, INDEP = {}, {}
print("\nIndependence-SE LEVEL contrasts (cell-15 construction; WV5 rows = published anchors):")
for other in INSTS[1:]:
    for w in W567:
        ca, sa = level(FULL["conf_military"], w)
        co, so = level(FULL[other], w)
        d, se = ca - co, float(np.sqrt(sa**2 + so**2))
        POINT[(other, w)] = d
        INDEP[(other, w)] = se
        p_ = 2 * (1 - stats.norm.cdf(abs(d / se)))
        print(f"  army vs {SHORT[other]:<14} {w}: {d:+.4f}  SE {se:.4f}  p {p_:.4f}")
        add("INDEPENDENCE", f"army_vs_{SHORT[other]}", w, "LEVEL_DIFF", d, se, p=p_,
            notes="conservative upper-bound SE; WV5 anchors: parl p=.068, gov p=.020 (published)")
assert abs(POINT[("conf_parliament", "WV5")] - 0.398) < 0.005
assert abs(POINT[("conf_government", "WV5")] - 0.507) < 0.005
print("anchor check vs final_checks cell 15: PASSED")


GATE refit nr_x_WV4 = +0.716155 (SE 0.182816) n=8915 | published +0.716155 (0.182816) n=8915
REPLICATION GATE PASSED
common sample n = 8777; conf_civil_service non-missing = 8711
wave n: {'WV2': 712, 'WV3': 1250, 'WV4': 2642, 'WV5': 1047, 'WV6': 1314, 'WV7': 1812}



LEVELs and INTERs (common sample, ordinal, delta-method SE):
  military       n=8777  WV5 -0.0885(0.1640)  WV6 -0.3320(0.1385)  WV7 -0.0807(0.0966)
  parliament     n=8777  WV5 -0.4862(0.1430)  WV6 -0.4012(0.1314)  WV7 -0.0764(0.0891)
  government     n=8777  WV5 -0.5959(0.1443)  WV6 -0.4733(0.1318)  WV7 -0.5550(0.0903)
  civil_service  n=8711  WV5 -0.2940(0.1460)  WV6 -0.5409(0.1347)  WV7 -0.4327(0.0936)

Independence-SE LEVEL contrasts (cell-15 construction; WV5 rows = published anchors):
  army vs parliament     WV5: +0.3976  SE 0.2176  p 0.0677
  army vs parliament     WV6: +0.0691  SE 0.1909  p 0.7173
  army vs parliament     WV7: -0.0044  SE 0.1315  p 0.9734
  army vs government     WV5: +0.5073  SE 0.2185  p 0.0202
  army vs government     WV6: +0.1413  SE 0.1912  p 0.4600
  army vs government     WV7: +0.4743  SE 0.1323  p 0.0003
  army vs civil_service  WV5: +0.2055  SE 0.2196  p 0.3494
  army vs civil_service  WV6: +0.2088  SE 0.1932  p 0.2798
  army vs civil_service  WV7: +

## Check A — respondent-level cluster bootstrap (primary)

Nonparametric bootstrap resampling **respondents** with replacement (the whole respondent, so the
cross-outcome correlation is preserved), **stratified by wave** (each wave resampled to its own n).
Per replicate the exact layer4b ordinal spec is refit on all four institutions. Fixed seed
20260716. **Fast fitter:** `OrderedModel` differentiates numerically (~3 s per fit → ~12 s per
replicate → 210 min for B = 1,000, over the ~90-min budget). The bootstrap therefore uses a custom
implementation of the *identical* ordered-logit likelihood (same parameterization: β, first
threshold, log-increments) with an analytic gradient — verified below to (i) reproduce
`OrderedModel`'s log-likelihood exactly, (ii) match a tightened `OrderedModel` (gtol=1e-9) optimum
to <1e-6, on the full sample and on a resampled replicate. Convergence per fit = gradient
inf-norm < 1e-3 (BFGS's `success` flag misreports "precision loss" at gradients ~1e-5).
**CIs:** percentile and bias-corrected BC (a = 0). Full BCa needs a jackknife pass (8,777 refits
× 4 institutions) — omitted as not cheap. Percentile p resolution at B = 1,000 is ~0.002.

In [2]:
# fast ordered-logit (analytic gradient) + verification gates
K = 4
def negll_grad(theta, X, y0):
    p = X.shape[1]
    beta, c1, d = theta[:p], theta[p], theta[p + 1:]
    ed = np.exp(d)
    a = np.concatenate([[c1], c1 + np.cumsum(ed)])
    eta = X @ beta
    a_full = np.concatenate([[-np.inf], a, [np.inf]])
    z_hi = a_full[y0 + 1] - eta
    z_lo = a_full[y0] - eta
    F_hi = np.where(np.isinf(z_hi), 1.0, expit(z_hi))
    F_lo = np.where(np.isinf(z_lo), 0.0, expit(z_lo))
    f_hi = np.where(np.isinf(z_hi), 0.0, F_hi * (1 - F_hi))
    f_lo = np.where(np.isinf(z_lo), 0.0, F_lo * (1 - F_lo))
    prob = np.clip(F_hi - F_lo, 1e-300, None)
    ll = np.log(prob).sum()
    inv = 1.0 / prob
    g_eta = -(f_hi - f_lo) * inv
    g_beta = X.T @ g_eta
    g_alpha = np.zeros(K - 1)
    wa_hi = f_hi * inv
    wa_lo = f_lo * inv
    for j in range(1, K):
        g_alpha[j - 1] = wa_hi[y0 + 1 == j].sum() - wa_lo[y0 == j].sum()
    g_c1 = g_alpha.sum()
    g_d = ed * np.array([g_alpha[j:].sum() for j in range(1, K - 1)])
    return -ll, -np.concatenate([g_beta, [g_c1], g_d])

def fast_fit(X, y0, x0):
    r = optimize.minimize(negll_grad, x0, args=(X, y0), jac=True,
                          method="BFGS", options={"maxiter": 400, "gtol": 1e-6})
    _, gr = negll_grad(r.x, X, y0)
    return r.x, float(np.max(np.abs(gr)))

print("verification vs OrderedModel:")
worst_llf, worst_opt = 0.0, 0.0
for inst in INSTS:
    m = ~np.isnan(Y[inst])
    Xi, y0 = X_pred[m], Y[inst][m].astype(int) - 1
    theta_def = FULL[inst].params.values
    nll, _ = negll_grad(theta_def, Xi, y0)
    worst_llf = max(worst_llf, abs(FULL[inst].llf + nll))
    tight = OrderedModel(pd.Series(y0 + 1), pd.DataFrame(Xi, columns=PREDICTORS),
                         distr="logit").fit(method="bfgs", maxiter=800, disp=False, gtol=1e-9)
    th, ginf = fast_fit(Xi, y0, theta_def)
    dmax = float(np.max(np.abs(th - tight.params.values)))
    worst_opt = max(worst_opt, dmax)
    print(f"  {SHORT[inst]:<14} llf diff {abs(FULL[inst].llf + nll):.2e}; "
          f"|custom - sm(gtol=1e-9)| max {dmax:.2e}; grad_inf {ginf:.2e}")
assert worst_llf < 1e-6 and worst_opt < 1e-5, "fast-fitter verification FAILED — STOP"
# resample cross-check (separate rng; does not touch the bootstrap seed)
idx_by_wave = {w: np.where(common["wave"].values == w)[0] for w in WAVES}
rngx = np.random.default_rng(99)
idxx = np.concatenate([rngx.choice(idx_by_wave[w], size=len(idx_by_wave[w]), replace=True)
                       for w in WAVES])
yv = Y["conf_military"][idxx]
mm = ~np.isnan(yv)
thx, _ = fast_fit(X_pred[idxx[mm]], yv[mm].astype(int) - 1, FULL["conf_military"].params.values)
smx = OrderedModel(pd.Series(yv[mm].astype(int)),
                   pd.DataFrame(X_pred[idxx[mm]], columns=PREDICTORS),
                   distr="logit").fit(method="bfgs", maxiter=800, disp=False, gtol=1e-9)
dres = float(np.max(np.abs(thx - smx.params.values)))
print(f"resampled-replicate cross-check (army): max|diff| = {dres:.2e}")
assert dres < 1e-5, "resample cross-check FAILED — STOP"
print("FAST FITTER VERIFIED")


verification vs OrderedModel:


  military       llf diff 0.00e+00; |custom - sm(gtol=1e-9)| max 1.35e-07; grad_inf 1.12e-07


  parliament     llf diff 0.00e+00; |custom - sm(gtol=1e-9)| max 3.10e-08; grad_inf 9.54e-09


  government     llf diff 0.00e+00; |custom - sm(gtol=1e-9)| max 3.88e-08; grad_inf 9.18e-07


  civil_service  llf diff 0.00e+00; |custom - sm(gtol=1e-9)| max 8.08e-08; grad_inf 9.09e-07


resampled-replicate cross-check (army): max|diff| = 7.27e-08
FAST FITTER VERIFIED


In [3]:
# A2 timing protocol + full bootstrap
SEED, B = 20260716, 1000
I_NR = PREDICTORS.index("not_religious_dummy")
I_W = {w: PREDICTORS.index(f"nr_x_{w}") for w in W567}
START = {inst: FULL[inst].params.values for inst in INSTS}

# time ONE OrderedModel-based replicate (separate rng; protocol requirement)
rngt = np.random.default_rng(12345)
idxt = np.concatenate([rngt.choice(idx_by_wave[w], size=len(idx_by_wave[w]), replace=True)
                       for w in WAVES])
t0 = time.time()
for inst in INSTS:
    yv = Y[inst][idxt]
    m = ~np.isnan(yv)
    OrderedModel(pd.Series(yv[m].astype(int)),
                 pd.DataFrame(X_pred[idxt[m]], columns=PREDICTORS),
                 distr="logit").fit(method="bfgs", maxiter=400, disp=False,
                                    start_params=START[inst])
t_sm = time.time() - t0
t0 = time.time()
for inst in INSTS:
    yv = Y[inst][idxt]
    m = ~np.isnan(yv)
    fast_fit(X_pred[idxt[m]], yv[m].astype(int) - 1, START[inst])
t_fast = time.time() - t0
print(f"one replicate, OrderedModel path: {t_sm:.1f}s -> projected B=1000: "
      f"{t_sm*1000/60:.0f} min, B=500: {t_sm*500/60:.0f} min")
print(f"one replicate, verified fast path: {t_fast:.2f}s -> projected B=1000: "
      f"{t_fast*1000/60:.1f} min")
print(f"DECISION: OrderedModel path exceeds the ~90-min budget at both B=1000 and B=500; "
      f"the verified fast path makes B = {B} feasible -> B = {B} used (seed {SEED})")

rng = np.random.default_rng(SEED)
levels_b = np.full((B, len(INSTS), 3), np.nan)
bad = 0
t0 = time.time()
for b in range(B):
    idx = np.concatenate([rng.choice(idx_by_wave[w], size=len(idx_by_wave[w]),
                                     replace=True) for w in WAVES])
    ok = True
    vals = np.empty((len(INSTS), 3))
    for i, inst in enumerate(INSTS):
        yv = Y[inst][idx]
        m = ~np.isnan(yv)
        th, ginf = fast_fit(X_pred[idx[m]], yv[m].astype(int) - 1, START[inst])
        if ginf >= 1e-3:
            ok = False
            break
        for j, w in enumerate(W567):
            vals[i, j] = th[I_NR] + th[I_W[w]]
    if ok:
        levels_b[b] = vals
    else:
        bad += 1
good = ~np.isnan(levels_b[:, 0, 0])
LB = levels_b[good]
B_used = int(LB.shape[0])
print(f"bootstrap complete: B={B}, non-converged dropped={bad}, used={B_used} "
      f"[{(time.time()-t0)/60:.1f} min]")


one replicate, OrderedModel path: 14.7s -> projected B=1000: 246 min, B=500: 123 min
one replicate, verified fast path: 0.14s -> projected B=1000: 2.3 min
DECISION: OrderedModel path exceeds the ~90-min budget at both B=1000 and B=500; the verified fast path makes B = 1000 feasible -> B = 1000 used (seed 20260716)


bootstrap complete: B=1000, non-converged dropped=5, used=995 [2.4 min]


In [4]:
# A3 contrasts, A4 diagnostic, A5 joint Wald
def pctl_p(d):
    lo = (1 + (d <= 0).sum()) / (len(d) + 1)
    hi = (1 + (d >= 0).sum()) / (len(d) + 1)
    return min(1.0, 2 * min(lo, hi))

def bc_ci(d, dhat, alpha=0.05):
    z0 = stats.norm.ppf(((d < dhat).sum()) / len(d))
    a1 = stats.norm.cdf(2 * z0 + stats.norm.ppf(alpha / 2))
    a2 = stats.norm.cdf(2 * z0 + stats.norm.ppf(1 - alpha / 2))
    return float(np.quantile(d, a1)), float(np.quantile(d, a2))

ROLE = {"conf_parliament": "PRIMARY", "conf_government": "POSITIVE CONTROL",
        "conf_civil_service": "SECONDARY"}
tabA = []
for oi, other in enumerate(INSTS[1:], start=1):
    for j, w in enumerate(W567):
        d = LB[:, 0, j] - LB[:, oi, j]
        pt = POINT[(other, w)]
        se_b = float(d.std(ddof=1))
        ci = (float(np.quantile(d, .025)), float(np.quantile(d, .975)))
        bc = bc_ci(d, pt)
        p_ = pctl_p(d)
        cr = float(np.corrcoef(LB[:, 0, j], LB[:, oi, j])[0, 1])
        tabA.append({"contrast": f"army vs {SHORT[other]}", "wave": w,
                     "point(full fit)": f"{pt:+.4f}", "boot SE": f"{se_b:.4f}",
                     "indep SE": f"{INDEP[(other, w)]:.4f}",
                     "SE ratio": f"{se_b/INDEP[(other, w)]:.3f}",
                     "pct 95% CI": f"[{ci[0]:+.4f}, {ci[1]:+.4f}]",
                     "BC 95% CI": f"[{bc[0]:+.4f}, {bc[1]:+.4f}]",
                     "p_pct": f"{p_:.4f}", "corr(army,other)": f"{cr:+.3f}"})
        add("BOOTSTRAP", f"army_vs_{SHORT[other]}", w, "LEVEL_DIFF", pt, se_b, ci=ci,
            p=p_, n_boot=B_used,
            notes=f"{ROLE[other]}; percentile p; BC CI [{bc[0]:+.4f},{bc[1]:+.4f}]; "
                  f"boot corr {cr:+.3f}; indep SE {INDEP[(other, w)]:.4f}")
        add("BOOTSTRAP", f"army_vs_{SHORT[other]}", w, "BOOT_CORR", cr, n_boot=B_used,
            notes="corr of bootstrap LEVEL estimates (A4 diagnostic)")
print(pd.DataFrame(tabA).to_string(index=False))

print("\nA5 — joint persistence Wald (3 df, bootstrap covariance of the difference vector):")
for other in ["conf_parliament", "conf_government", "conf_civil_service"]:
    oi = INSTS.index(other)
    D = LB[:, 0, :] - LB[:, oi, :]
    dvec = np.array([POINT[(other, w)] for w in W567])
    V = np.cov(D.T)
    W_ = float(dvec @ np.linalg.inv(V) @ dvec)
    pW = float(1 - stats.chi2.cdf(W_, 3))
    print(f"  army vs {SHORT[other]:<12}: chi2(3) = {W_:.2f}   p = {pW:.5f}")
    add("BOOTSTRAP", f"army_vs_{SHORT[other]}", "WV5-WV7", "JOINT_WALD_CHI2_3DF", W_,
        p=pW, n_boot=B_used, notes="H0: all three LEVEL differences zero")


             contrast wave point(full fit) boot SE indep SE SE ratio         pct 95% CI          BC 95% CI  p_pct corr(army,other)
   army vs parliament  WV5         +0.3976  0.1958   0.2176    0.900 [+0.0150, +0.7931] [+0.0605, +0.8261] 0.0422           +0.254
   army vs parliament  WV6         +0.0691  0.1467   0.1909    0.768 [-0.1966, +0.3746] [-0.2130, +0.3662] 0.6145           +0.357
   army vs parliament  WV7         -0.0044  0.0919   0.1315    0.699 [-0.1819, +0.1879] [-0.1769, +0.1887] 0.9438           +0.376
   army vs government  WV5         +0.5073  0.2014   0.2185    0.922 [+0.1287, +0.9101] [+0.1437, +0.9375] 0.0181           +0.217
   army vs government  WV6         +0.1413  0.1703   0.1912    0.891 [-0.1730, +0.4748] [-0.1719, +0.4756] 0.3916           +0.130
   army vs government  WV7         +0.4743  0.0933   0.1323    0.705 [+0.2805, +0.6553] [+0.2778, +0.6529] 0.0020           +0.382
army vs civil_service  WV5         +0.2055  0.2169   0.2196    0.988 [-0.1810, +0.6

## Check B — stacked binary logit, respondent-clustered SEs (companion)

Long format (respondent × institution), DV = high confidence (categories 3–4), design fully
institution-interacted (institution × every layer4b term incl. intercept), one binary logit,
`cov_type='cluster'` on respondent id. Because the likelihood is separable by institution, the
point estimates must equal four separate logits (B2 gate) and the *nonrobust* covariance is
block-diagonal — its contrast SE is exactly the independence SE on this route (B4).
**B5: this uses the binary recode and is therefore a different estimand from Check A — a
consistency check on sign and significance, not a numeric match.**

In [5]:
# stacked build + B2 gate + B3/B4 contrasts
common["rid"] = np.arange(len(common))
long_parts = []
for inst in INSTS:
    d = common[common[inst].notna()].copy()
    d["y_high"] = (d[inst] >= 3).astype(int)
    d["institution"] = inst
    long_parts.append(d[["rid", "institution", "y_high"] + PREDICTORS])
longdf = pd.concat(long_parts, ignore_index=True)
Xcols = {}
for inst in INSTS:
    ind = (longdf["institution"] == inst).astype(float)
    Xcols[f"{inst}:const"] = ind
    for p_ in PREDICTORS:
        Xcols[f"{inst}:{p_}"] = ind * longdf[p_]
Xl = pd.DataFrame(Xcols)
yl = longdf["y_high"]
res_plain = sm.Logit(yl, Xl).fit(disp=False, maxiter=400)
res_clu = sm.Logit(yl, Xl).fit(disp=False, maxiter=400, cov_type="cluster",
                               cov_kwds={"groups": longdf["rid"].values})
assert res_plain.mle_retvals["converged"] and res_clu.mle_retvals["converged"]
print(f"long rows = {len(longdf)}; params plain==clustered: "
      f"{bool(np.allclose(res_plain.params.values, res_clu.params.values))}")

maxdiff = 0.0
for inst in INSTS:
    d = common[common[inst].notna()]
    rsep = sm.Logit((d[inst] >= 3).astype(int),
                    sm.add_constant(d[PREDICTORS])).fit(disp=False, maxiter=400)
    sv = np.array([float(res_plain.params[f"{inst}:const"])] +
                  [float(res_plain.params[f"{inst}:{p_}"]) for p_ in PREDICTORS])
    pv = np.array([float(rsep.params["const"])] +
                  [float(rsep.params[p_]) for p_ in PREDICTORS])
    maxdiff = max(maxdiff, float(np.max(np.abs(sv - pv))))
print(f"B2 GATE: max |stacked - separate| = {maxdiff:.2e}")
assert maxdiff < 1e-4, "B2 GATE FAILED: design matrix wrong — STOP"

def Lvec(a, b, w):
    idx = list(res_plain.params.index)
    L = np.zeros(len(idx))
    L[idx.index(f"{a}:not_religious_dummy")] = 1
    L[idx.index(f"{a}:nr_x_{w}")] = 1
    L[idx.index(f"{b}:not_religious_dummy")] = -1
    L[idx.index(f"{b}:nr_x_{w}")] = -1
    return L

Vclu = np.asarray(res_clu.cov_params())
Vnaive = np.asarray(res_plain.cov_params())
theta = res_clu.params.values
STACKED = {}
print("\nB3/B4 — binary LEVEL contrasts (clustered vs naive/independence SE):")
for other in INSTS[1:]:
    Ls = []
    for w in W567:
        L = Lvec("conf_military", other, w)
        Ls.append(L)
        est = float(L @ theta)
        se_c = float(np.sqrt(L @ Vclu @ L))
        se_n = float(np.sqrt(L @ Vnaive @ L))
        p_c = 2 * (1 - stats.norm.cdf(abs(est / se_c)))
        STACKED[(other, w)] = (est, se_c, p_c)
        print(f"  army vs {SHORT[other]:<14} {w}: {est:+.4f}  SE_clu {se_c:.4f} "
              f"(z {est/se_c:+.2f}, p {p_c:.4f})  SE_naive {se_n:.4f}  ratio {se_c/se_n:.3f}")
        add("STACKED_CLUSTERED", f"army_vs_{SHORT[other]}", w, "LEVEL_DIFF_BINARY",
            est, se_c, p=p_c, notes=f"naive/indep SE {se_n:.4f} (ratio {se_c/se_n:.3f})")
    A = np.vstack(Ls)
    dv = A @ theta
    W_ = float(dv @ np.linalg.inv(A @ Vclu @ A.T) @ dv)
    pW = float(1 - stats.chi2.cdf(W_, 3))
    STACKED[(other, "joint")] = (W_, np.nan, pW)
    print(f"    joint chi2(3) = {W_:.2f}  p = {pW:.5f}")
    add("STACKED_CLUSTERED", f"army_vs_{SHORT[other]}", "WV5-WV7", "JOINT_WALD_CHI2_3DF",
        W_, p=pW)


long rows = 35042; params plain==clustered: True


B2 GATE: max |stacked - separate| = 7.11e-15

B3/B4 — binary LEVEL contrasts (clustered vs naive/independence SE):
  army vs parliament     WV5: +0.3220  SE_clu 0.2243 (z +1.44, p 0.1512)  SE_naive 0.2684  ratio 0.836
  army vs parliament     WV6: +0.1502  SE_clu 0.1806 (z +0.83, p 0.4055)  SE_naive 0.2271  ratio 0.795
  army vs parliament     WV7: +0.0379  SE_clu 0.1623 (z +0.23, p 0.8155)  SE_naive 0.1872  ratio 0.867
    joint chi2(3) = 2.72  p = 0.43750
  army vs government     WV5: +0.5527  SE_clu 0.2366 (z +2.34, p 0.0195)  SE_naive 0.2725  ratio 0.868
  army vs government     WV6: +0.3683  SE_clu 0.1983 (z +1.86, p 0.0633)  SE_naive 0.2306  ratio 0.860
  army vs government     WV7: +0.6192  SE_clu 0.1664 (z +3.72, p 0.0002)  SE_naive 0.1908  ratio 0.872
    joint chi2(3) = 21.89  p = 0.00007
  army vs civil_service  WV5: +0.0741  SE_clu 0.2330 (z +0.32, p 0.7505)  SE_naive 0.2650  ratio 0.879
  army vs civil_service  WV6: +0.2330  SE_clu 0.1886 (z +1.24, p 0.2167)  SE_naive 0.22

## Check C — weighted sensitivity (Check B only)

Positional per-wave weight merge as validated in `preflight_refits.ipynb` (V236/V245/V259/V258
for WV3–WV6; V376/W_WEIGHT ≡ 1.0 for WV2/WV7). Weighted binary logit (GLM Binomial,
`var_weights`), respondent-clustered sandwich; statsmodels' clustered covariance under
`var_weights` is cross-checked against a manually computed cluster sandwich A⁻¹BA⁻¹ built from
the weighted per-observation scores.

In [6]:
# weighted stacked logit
FILES = {"WV2": "F00008272-WV2_Data_Turkey_Excel_v1.6.1.xlsx",
         "WV3": "F00008135-WV3_Data_Turkey_Excel_v20221107.1.xlsx",
         "WV4": "F00008022-WV4_Data_Turkey_Excel_v20201117.1.xlsx",
         "WV5": "F00007873-WV5_Data_Turkey_Excel_v20201117.1.xlsx",
         "WV6": "F00007629-WV6_Data_Turkey_2011_Excel_v20180912.xlsx",
         "WV7": "F00013222-WVS_Wave_7_Turkey_Excel_v5.1.xlsx"}
WCODE = {"WV2": "V376", "WV3": "V236", "WV4": "V245",
         "WV5": "V259", "WV6": "V258", "WV7": "W_WEIGHT"}
parts = []
for w in WAVES:
    hm = {str(c).split(":", 1)[0].strip(): c
          for c in pd.read_excel(FILES[w], nrows=0).columns}
    src = pd.read_excel(FILES[w], usecols=[hm[WCODE[w]]]).rename(
        columns={hm[WCODE[w]]: "cal_weight"})
    src["wave"] = w
    src["row_in_wave"] = np.arange(len(src))
    parts.append(src)
common_w = common.merge(pd.concat(parts, ignore_index=True),
                        on=["wave", "row_in_wave"], how="left")
assert common_w["cal_weight"].notna().all()
wl = longdf.merge(common_w[["rid", "cal_weight"]], on="rid", how="left")
glm_w = sm.GLM(yl, Xl, family=sm.families.Binomial(),
               var_weights=wl["cal_weight"].values).fit(
    cov_type="cluster", cov_kwds={"groups": wl["rid"].values}, maxiter=200)
assert glm_w.converged
# manual cluster sandwich cross-check
Xn = Xl.to_numpy()
wv = wl["cal_weight"].values
mu = glm_w.fittedvalues.values if hasattr(glm_w.fittedvalues, "values") else np.asarray(glm_w.fittedvalues)
A = (Xn * (wv * mu * (1 - mu))[:, None]).T @ Xn
Sobs = pd.DataFrame(Xn * (wv * (yl.values - mu))[:, None])
Sc = Sobs.groupby(wl["rid"].values).sum().to_numpy()
cov_man = np.linalg.inv(A) @ (Sc.T @ Sc) @ np.linalg.inv(A)
Vw = np.asarray(glm_w.cov_params())
rel = np.max(np.abs(cov_man - Vw)) / np.max(np.abs(Vw))
print(f"manual vs statsmodels clustered cov: max rel diff = {rel:.2e} "
      f"({'agree — statsmodels used' if rel < 0.02 else 'DISAGREE — manual used'})")
VW = Vw if rel < 0.02 else cov_man
thw = glm_w.params.values
WSTACKED = {}
print("\nC1 — weighted vs unweighted clustered contrasts:")
for other in INSTS[1:]:
    Ls = []
    for w in W567:
        L = Lvec("conf_military", other, w)
        Ls.append(L)
        est = float(L @ thw)
        se = float(np.sqrt(L @ VW @ L))
        p_ = 2 * (1 - stats.norm.cdf(abs(est / se)))
        WSTACKED[(other, w)] = (est, se, p_)
        u = STACKED[(other, w)]
        print(f"  army vs {SHORT[other]:<14} {w}: wtd {est:+.4f} ({se:.4f}) p {p_:.4f}   "
              f"unw {u[0]:+.4f} ({u[1]:.4f}) p {u[2]:.4f}")
        add("STACKED_WEIGHTED", f"army_vs_{SHORT[other]}", w, "LEVEL_DIFF_BINARY",
            est, se, p=p_, notes="calibration weights; cluster sandwich")
    A3_ = np.vstack(Ls)
    dv = A3_ @ thw
    W_ = float(dv @ np.linalg.inv(A3_ @ VW @ A3_.T) @ dv)
    pW = float(1 - stats.chi2.cdf(W_, 3))
    WSTACKED[(other, "joint")] = (W_, np.nan, pW)
    print(f"    joint chi2(3) = {W_:.2f}  p = {pW:.5f}")
    add("STACKED_WEIGHTED", f"army_vs_{SHORT[other]}", "WV5-WV7", "JOINT_WALD_CHI2_3DF",
        W_, p=pW)


manual vs statsmodels clustered cov: max rel diff = 2.37e-03 (agree — statsmodels used)

C1 — weighted vs unweighted clustered contrasts:
  army vs parliament     WV5: wtd +0.3358 (0.2304) p 0.1451   unw +0.3220 (0.2243) p 0.1512
  army vs parliament     WV6: wtd +0.1367 (0.1870) p 0.4646   unw +0.1502 (0.1806) p 0.4055
  army vs parliament     WV7: wtd +0.0416 (0.1632) p 0.7986   unw +0.0379 (0.1623) p 0.8155
    joint chi2(3) = 2.62  p = 0.45390
  army vs government     WV5: wtd +0.5707 (0.2441) p 0.0194   unw +0.5527 (0.2366) p 0.0195
  army vs government     WV6: wtd +0.3820 (0.2035) p 0.0605   unw +0.3683 (0.1983) p 0.0633
  army vs government     WV7: wtd +0.6238 (0.1675) p 0.0002   unw +0.6192 (0.1664) p 0.0002
    joint chi2(3) = 21.78  p = 0.00007
  army vs civil_service  WV5: wtd +0.1194 (0.2362) p 0.6131   unw +0.0741 (0.2330) p 0.7505
  army vs civil_service  WV6: wtd +0.1613 (0.1952) p 0.4086   unw +0.2330 (0.1886) p 0.2167
  army vs civil_service  WV7: wtd +0.4733 (0.1659

## Summary — every route to the same question

Ordinal columns (independence, bootstrap) and binary columns (clustered, weighted) are different
estimands (B5); compare sign and significance, not magnitudes.

In [7]:
# summary table + tidy CSV
mrows = []
master = pd.DataFrame(ROWS, columns=["method", "contrast", "wave", "quantity",
                                     "estimate", "se", "ci_lo", "ci_hi", "p",
                                     "n_boot", "notes"])
mdf = master.set_index(["method", "contrast", "wave", "quantity"])
def getp(method, contrast, wave, quantity):
    try:
        return float(mdf.loc[(method, contrast, wave, quantity), "p"])
    except KeyError:
        return np.nan
for other in INSTS[1:]:
    c = f"army_vs_{SHORT[other]}"
    for w in W567:
        mrows.append({"contrast": c, "wave": w,
                      "indep_p (ordinal)": round(getp("INDEPENDENCE", c, w, "LEVEL_DIFF"), 4),
                      "bootstrap_p (ordinal)": round(getp("BOOTSTRAP", c, w, "LEVEL_DIFF"), 4),
                      "clustered_p (binary)": round(getp("STACKED_CLUSTERED", c, w, "LEVEL_DIFF_BINARY"), 4),
                      "weighted_clustered_p (binary)": round(getp("STACKED_WEIGHTED", c, w, "LEVEL_DIFF_BINARY"), 4)})
    if True:
        mrows.append({"contrast": c, "wave": "JOINT WV5-7",
                      "indep_p (ordinal)": np.nan,
                      "bootstrap_p (ordinal)": round(getp("BOOTSTRAP", c, "WV5-WV7", "JOINT_WALD_CHI2_3DF"), 5),
                      "clustered_p (binary)": round(getp("STACKED_CLUSTERED", c, "WV5-WV7", "JOINT_WALD_CHI2_3DF"), 5),
                      "weighted_clustered_p (binary)": round(getp("STACKED_WEIGHTED", c, "WV5-WV7", "JOINT_WALD_CHI2_3DF"), 5)})
master.to_csv(f"{RESULTS}/persistence_test_table.csv", index=False)
print(f"wrote results/persistence_test_table.csv ({len(master)} rows)\n")
pd.DataFrame(mrows)


wrote results/persistence_test_table.csv (79 rows)



,contrast,wave,indep_p (ordinal),bootstrap_p (ordinal),clustered_p (binary),weighted_clustered_p (binary)
0,army_vs_parliament,WV5,0.0677,0.04220,0.15120,0.14510
1,army_vs_parliament,WV6,0.7173,0.61450,0.40550,0.46460
2,army_vs_parliament,WV7,0.9734,0.94380,0.81550,0.79860
3,army_vs_parliament,JOINT WV5-7,NaN,0.22389,0.43750,0.45390
4,army_vs_government,WV5,0.0202,0.01810,0.01950,0.01940
5,army_vs_government,WV6,0.4600,0.39160,0.06330,0.06050
6,army_vs_government,WV7,0.0003,0.00200,0.00020,0.00020
7,army_vs_government,JOINT WV5-7,NaN,0.00000,0.00007,0.00007
8,army_vs_civil_service,WV5,0.3494,0.37550,0.75050,0.61310
9,army_vs_civil_service,WV6,0.2798,0.16270,0.21670,0.40860
